# 3-Fold CV: DINOv2 + LoRA + Linear Probe vs DINOv2 Frozen Baseline

Mirrors `02_cv_lora.ipynb` but with DINOv2 ViT-L/14 instead of BioCLIP 2 as the foundation model.

**Purpose:** complete the foundation-model adaptation comparison. Notebook 02 showed that LoRA improves BioCLIP 2 substantially on diagnostic classes (Hemiptera +14.4pp). This notebook tests whether the same improvement occurs for DINOv2 — does LoRA help equally regardless of pretraining domain, or is BioCLIP 2's biological pretraining specifically what LoRA adapts effectively?

**Hyperparameters:** match the winning sweep configuration for BioCLIP 2 (lr=5e-4, r=16, epochs=2) for direct comparability. Not separately tuned for DINOv2 — using the same config tests whether the BioCLIP 2 winning hyperparameters transfer to DINOv2.

**Outputs:**
- `output_dinov2_lora/per_fold_per_class_results.csv` — granular per-fold per-class metrics
- `output_dinov2_lora/per_class_summary_mean_std.csv` — per-class mean ± std across folds
- `output_dinov2_lora/per_class_delta.csv` — per-class delta (LoRA - Frozen)
- `output_dinov2_lora/headline_metrics_mean_std.csv` — aggregate metrics with mean ± std
- `output_dinov2_lora/features_fold{0,1,2}.npz` — cached frozen and LoRA features for downstream analysis

## Imports and configuration

In [1]:
import re
import time
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import normalize as sk_normalize
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from peft import LoraConfig, get_peft_model

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.8.0+cu128
CUDA: True
GPU: NVIDIA A100-SXM4-80GB


In [2]:
CONFIG = {
    # DINOv2 ViT-L/14 — matches BioCLIP 2's vision encoder size
    "dinov2_model": "dinov2_vitl14",
    "feature_dim": 1024,
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    "meta_path": "/workspace/thesis/ultimate_bioclip_top1_rank_order_meta_12062025.csv",
    "image_root": "/workspace/thesis/ALL_copy",
    "output_dir": "/workspace/thesis/output_dinov2_lora_without_qkv",

    "image_id_col": "image_id",
    "label_col": "label",

    "k_folds": 3,
    "random_state": 0,
    "min_class_size": 50,

    # ============================================
    # Hyperparameters matching BioCLIP 2 sweep winner are changes
    # ============================================
    "lr": 5e-4,
    "r": 16,
    "epochs": 2,
    # ============================================

    # DINOv2 LoRA targets — see Cell 8 for verification of module names
    # DINOv2 uses different naming than CLIP-family: typically 'qkv', 'proj', 'fc1', 'fc2'
    "lora_target_modules": ["fc1", "fc2"],
    "lora_alpha_factor": 2,
    "lora_dropout": 0.05,

    "batch_size": 64,
    "weight_decay": 1e-4,
    "num_workers": 4,

    "linear_probe_C": 10.0,
    "linear_probe_max_iter": 5000,

    # Image preprocessing (ImageNet conventions for DINOv2)
    "image_size": 224,
    "imagenet_mean": [0.485, 0.456, 0.406],
    "imagenet_std": [0.229, 0.224, 0.225],

    # Feature saving
    "save_features": True,
}

CROP_ID_PATTERN = re.compile(r"(\d{14}_crop_[a-zA-Z0-9]+\.jpg)$")
Path(CONFIG["output_dir"]).mkdir(parents=True, exist_ok=True)

print(f"Hyperparameters: lr={CONFIG['lr']}, r={CONFIG['r']}, epochs={CONFIG['epochs']}")
print(f"LoRA target modules: {CONFIG['lora_target_modules']}")
print(f"Output: {CONFIG['output_dir']}")
print(f"Save features: {CONFIG['save_features']}")

Hyperparameters: lr=0.0005, r=16, epochs=2
LoRA target modules: ['fc1', 'fc2']
Output: /workspace/thesis/output_dinov2_lora_without_qkv
Save features: True


## Load DINOv2

DINOv2 is loaded via `torch.hub` from Meta's official release. The model outputs the CLS token feature directly (1024-dim for ViT-L/14).

In [3]:
print(f"Loading DINOv2 {CONFIG['dinov2_model']}...")
model = torch.hub.load('facebookresearch/dinov2', CONFIG['dinov2_model'])
model = model.to(CONFIG["device"])
print(f"Loaded. Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading DINOv2 dinov2_vitl14...


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Loaded. Total parameters: 304,368,640


## Verify LoRA target module names

DINOv2 uses different layer naming than CLIP-family models. Common targets in DINOv2 ViT are:
- `qkv` — combined query/key/value projection
- `proj` — attention output projection
- `fc1`, `fc2` — MLP layers

This cell prints all Linear layers in the model to confirm the targets in CONFIG match the actual module structure.

In [4]:
# Showing first few Linear layer names to verify LoRA targets
linear_names = []
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        linear_names.append(name)

# Showing structure of first block
print("Linear layers in first transformer block:")
for name in linear_names[:10]:
    print(f"  {name}")

print(f"\nTotal Linear layers in model: {len(linear_names)}")

# Verifying LoRA targets exist
target_strings = CONFIG["lora_target_modules"]
print(f"\nVerifying LoRA targets {target_strings}:")
for target in target_strings:
    matches = [n for n in linear_names if n.endswith(f".{target}")]
    print(f"   '{target}' found in {len(matches)} layers")
    if matches:
        print(f"   Example: {matches[0]}")

Linear layers in first transformer block:
  blocks.0.attn.qkv
  blocks.0.attn.proj
  blocks.0.mlp.fc1
  blocks.0.mlp.fc2
  blocks.1.attn.qkv
  blocks.1.attn.proj
  blocks.1.mlp.fc1
  blocks.1.mlp.fc2
  blocks.2.attn.qkv
  blocks.2.attn.proj

Total Linear layers in model: 96

Verifying LoRA targets ['fc1', 'fc2']:
   'fc1' found in 24 layers
   Example: blocks.0.mlp.fc1
   'fc2' found in 24 layers
   Example: blocks.0.mlp.fc2


## Helper functions

In [5]:
def build_path_index(image_root):
    """Scanning local image directory, building {image_id: full_path}."""
    image_root = Path(image_root)
    files = list(image_root.glob("*.jpg"))
    path_index = {}
    for f in files:
        m = CROP_ID_PATTERN.search(f.name)
        if m:
            path_index[m.group(1)] = str(f)
    print(f"Mapped {len(path_index)} image_ids")
    return path_index


class ImageOrderDataset(Dataset):
    """Dataset of (image, label) pairs loaded from disk."""
    def __init__(self, image_paths, labels, preprocess):
        self.image_paths = image_paths
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.preprocess = preprocess

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        img = self.preprocess(img)
        return img, self.labels[idx]


# Preprocessing matches notebook 04_dinov2 (no augmentation for fair comparison)
preprocess = transforms.Compose([
    transforms.Resize(CONFIG["image_size"], interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(CONFIG["image_size"]),
    transforms.ToTensor(),
    transforms.Normalize(mean=CONFIG["imagenet_mean"], std=CONFIG["imagenet_std"]),
])

In [6]:
def train_dinov2_lora(model_base, train_loader, classnames, config, fold_idx):
    """Applying LoRA to DINOv2 and training with a classification head.
    
    Returns the LoRA-wrapped model.
    """
    device = config["device"]
    n_classes = len(classnames)

    lora_config = LoraConfig(
        r=config["r"],
        lora_alpha=config["r"] * config["lora_alpha_factor"],
        lora_dropout=config["lora_dropout"],
        target_modules=config["lora_target_modules"],
        bias="none",
    )
    
    # Wrapping the whole model (DINOv2 doesn't have a separate .visual like open_clip)
    model_lora = get_peft_model(model_base, lora_config).to(device)
    model_lora.print_trainable_parameters()

    head = nn.Linear(config["feature_dim"], n_classes).to(device)
    trainable = [p for p in model_lora.parameters() if p.requires_grad] + list(head.parameters())

    optimizer = torch.optim.AdamW(trainable, lr=config["lr"], weight_decay=config["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["epochs"])

    for epoch in range(config["epochs"]):
        model_lora.train()
        head.train()
        total_loss, n_batches = 0.0, 0
        ep_start = time.time()
        for batch_idx, (imgs, labels) in enumerate(train_loader):
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            features = F.normalize(model_lora(imgs), dim=-1)
            logits = head(features)
            loss = F.cross_entropy(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1
            if batch_idx % 100 == 0:
                print(f"    Fold {fold_idx+1} Ep {epoch+1} batch {batch_idx}/{len(train_loader)}: "
                      f"loss={loss.item():.4f}")
        scheduler.step()
        print(f"    Fold {fold_idx+1} Ep {epoch+1}: avg loss={total_loss/n_batches:.4f} "
              f"({(time.time()-ep_start)/60:.1f} min)")
    return model_lora


def extract_features(encoder, dataset, config):
    """Extracting L2-normalized features. Works for both frozen and LoRA-wrapped models."""
    device = config["device"]
    encoder.eval()
    loader = DataLoader(
        dataset, batch_size=config["batch_size"], shuffle=False,
        num_workers=config["num_workers"], pin_memory=True,
    )
    all_features = []
    with torch.no_grad():
        for imgs, _ in loader:
            features = F.normalize(encoder(imgs.to(device, non_blocking=True)), dim=-1)
            all_features.append(features.cpu().numpy())
    return np.concatenate(all_features, axis=0)


def fit_linear_probe(train_features, train_labels, val_features, config):
    """Fitting multinomial logistic regression."""
    clf = LogisticRegression(
        C=config["linear_probe_C"],
        max_iter=config["linear_probe_max_iter"],
        solver="lbfgs",
        random_state=config["random_state"],
    )
    clf.fit(train_features, train_labels)
    return clf.predict(val_features)

## Load metadata and align with local images

In [7]:
path_index = build_path_index(CONFIG["image_root"])

meta = pd.read_csv(CONFIG["meta_path"])
meta = meta.dropna(subset=[CONFIG["label_col"], CONFIG["image_id_col"]]).reset_index(drop=True)
meta["local_path"] = meta[CONFIG["image_id_col"]].map(path_index)
matched = meta.dropna(subset=["local_path"]).reset_index(drop=True)

y_str = matched[CONFIG["label_col"]].to_numpy()
image_paths = matched["local_path"].to_numpy()
classnames = sorted(np.unique(y_str).tolist())
class_to_idx = {c: i for i, c in enumerate(classnames)}
y = np.array([class_to_idx[s] for s in y_str])

class_counts = pd.Series(y_str).value_counts()
eligible_classes = class_counts[class_counts >= CONFIG["min_class_size"]].index.tolist()
eligible_class_indices = [class_to_idx[c] for c in eligible_classes]
eligible_class_set = set(eligible_classes)

print(f"Matched: {len(matched)} crops")
print(f"Classes ({len(classnames)}): {classnames}")
print(f"Eligible for LoRA training (n>=50): {eligible_classes}")

Mapped 48095 image_ids
Matched: 39787 crops
Classes (14): ['Araneae', 'Blattodea', 'Coleoptera', 'Diptera', 'Hemiptera', 'Hymenoptera', 'Ixodida', 'Lepidoptera', 'Mecoptera', 'Neuroptera', 'Orthoptera', 'Plecoptera', 'Raphidioptera', 'Trichoptera']
Eligible for LoRA training (n>=50): ['Diptera', 'Hymenoptera', 'Lepidoptera', 'Coleoptera', 'Hemiptera', 'Araneae', 'Neuroptera', 'Trichoptera', 'Plecoptera']


## Setting up 3-fold stratified CV

Same protocol as notebook 02 for direct comparability.

In [8]:
class_counts_arr = np.bincount(y, minlength=len(classnames))
splittable_mask = class_counts_arr[y] >= CONFIG["k_folds"]
splittable_indices = np.where(splittable_mask)[0]
unsplittable_indices = np.where(~splittable_mask)[0]

print(f"Splittable samples (n>={CONFIG['k_folds']}): {len(splittable_indices)}")
print(f"Unsplittable samples (kept in train): {len(unsplittable_indices)}")

skf = StratifiedKFold(n_splits=CONFIG["k_folds"], shuffle=True, random_state=CONFIG["random_state"])
raw_splits = list(skf.split(splittable_indices, y[splittable_indices]))

fold_splits = []
for train_rel, val_rel in raw_splits:
    train_idx = np.concatenate([splittable_indices[train_rel], unsplittable_indices])
    val_idx = splittable_indices[val_rel]
    fold_splits.append((train_idx, val_idx))

for i, (tr, va) in enumerate(fold_splits):
    print(f"  Fold {i+1}: train={len(tr)}, val={len(va)}")

Splittable samples (n>=3): 39784
Unsplittable samples (kept in train): 3
  Fold 1: train=26525, val=13262
  Fold 2: train=26526, val=13261
  Fold 3: train=26526, val=13261


## Running cross-validation

For each fold:
1. Frozen baseline: extract features with frozen DINOv2, fit linear probe.
2. LoRA: reload fresh DINOv2, apply LoRA, train on n≥50 classes, extract features, fit linear probe.

Both methods evaluated on the same val set in each fold for within-fold comparison.

Features saved to `features_fold{0,1,2}.npz` for downstream analysis (PCA, etc.).

In [9]:
all_results = []
overall_start = time.time()

# Loading frozen model once (reused across folds for baseline)
print("Loading frozen DINOv2 (reused across folds for baseline)...")
model_frozen = torch.hub.load('facebookresearch/dinov2', CONFIG['dinov2_model'])
model_frozen = model_frozen.to(CONFIG["device"])
model_frozen.eval()

for fold_idx, (train_idx, val_idx) in enumerate(fold_splits):
    print(f"\n{'='*70}\nFOLD {fold_idx+1}/{CONFIG['k_folds']}\n{'='*70}")
    fold_start = time.time()

    train_mask = np.isin(y[train_idx], eligible_class_indices)
    train_idx_eligible = train_idx[train_mask]

    train_paths_eligible = image_paths[train_idx_eligible]
    train_labels_eligible = y[train_idx_eligible]
    train_paths_full = image_paths[train_idx]
    train_labels_full = y[train_idx]
    val_paths = image_paths[val_idx]
    val_labels = y[val_idx]

    train_dataset_eligible = ImageOrderDataset(train_paths_eligible, train_labels_eligible, preprocess)
    train_dataset_full = ImageOrderDataset(train_paths_full, train_labels_full, preprocess)
    val_dataset = ImageOrderDataset(val_paths, val_labels, preprocess)
    train_loader = DataLoader(train_dataset_eligible, batch_size=CONFIG["batch_size"],
                              shuffle=True, num_workers=CONFIG["num_workers"], pin_memory=True)

    print(f"  Train (LoRA, eligible): {len(train_idx_eligible)}")
    print(f"  Train (full, for probe): {len(train_idx)}")
    print(f"  Val: {len(val_idx)}")

    # ---- Frozen baseline ----
    print(f"\n  --- Frozen baseline ---")
    baseline_start = time.time()
    train_features_frozen = extract_features(model_frozen, train_dataset_full, CONFIG)
    val_features_frozen = extract_features(model_frozen, val_dataset, CONFIG)
    train_features_frozen = sk_normalize(train_features_frozen, norm="l2", axis=1)
    val_features_frozen = sk_normalize(val_features_frozen, norm="l2", axis=1)
    preds_frozen = fit_linear_probe(train_features_frozen, train_labels_full, val_features_frozen, CONFIG)
    print(f"  Baseline time: {timedelta(seconds=int(time.time() - baseline_start))}")

    report_frozen = classification_report(
        val_labels, preds_frozen, labels=list(range(len(classnames))),
        target_names=classnames, output_dict=True, zero_division=0,
    )
    for cls_name in classnames:
        stats = report_frozen.get(cls_name, {})
        all_results.append({
            "method": "frozen_linear_probe", "fold": fold_idx, "class": cls_name,
            "precision": round(stats.get("precision", 0.0), 4),
            "recall": round(stats.get("recall", 0.0), 4),
            "f1": round(stats.get("f1-score", 0.0), 4),
            "support": int(stats.get("support", 0)),
            "trained_in_lora": cls_name in eligible_class_set,
        })

    # ---- DINOv2 + LoRA + linear probe ----
    print(f"\n  --- DINOv2 + LoRA + linear probe ---")
    lora_start = time.time()
    print("  Reloading fresh DINOv2 for clean LoRA application...")
    reload_start = time.time()
    model_fresh = torch.hub.load('facebookresearch/dinov2', CONFIG['dinov2_model'])
    model_fresh = model_fresh.to(CONFIG["device"])
    print(f"    Reload took {time.time() - reload_start:.1f}s")
    
    model_lora = train_dinov2_lora(model_fresh, train_loader, classnames, CONFIG, fold_idx)
    train_features_lora = extract_features(model_lora, train_dataset_full, CONFIG)
    val_features_lora = extract_features(model_lora, val_dataset, CONFIG)
    train_features_lora = sk_normalize(train_features_lora, norm="l2", axis=1)
    val_features_lora = sk_normalize(val_features_lora, norm="l2", axis=1)
    preds_lora = fit_linear_probe(train_features_lora, train_labels_full, val_features_lora, CONFIG)
    print(f"  LoRA time: {timedelta(seconds=int(time.time() - lora_start))}")

    report_lora = classification_report(
        val_labels, preds_lora, labels=list(range(len(classnames))),
        target_names=classnames, output_dict=True, zero_division=0,
    )
    for cls_name in classnames:
        stats = report_lora.get(cls_name, {})
        all_results.append({
            "method": "lora_linear_probe", "fold": fold_idx, "class": cls_name,
            "precision": round(stats.get("precision", 0.0), 4),
            "recall": round(stats.get("recall", 0.0), 4),
            "f1": round(stats.get("f1-score", 0.0), 4),
            "support": int(stats.get("support", 0)),
            "trained_in_lora": cls_name in eligible_class_set,
        })

    # ---- Saving features for downstream analysis ----
    if CONFIG["save_features"]:
        feat_path = Path(CONFIG["output_dir"]) / f"features_fold{fold_idx}.npz"
        np.savez_compressed(
            feat_path,
            train_features_frozen=train_features_frozen,
            val_features_frozen=val_features_frozen,
            train_features_lora=train_features_lora,
            val_features_lora=val_features_lora,
            train_labels=train_labels_full,
            val_labels=val_labels,
            train_idx=train_idx,
            val_idx=val_idx,
            classnames=np.array(classnames),
            eligible_classes=np.array(eligible_classes),
        )
        size_mb = feat_path.stat().st_size / (1024 * 1024)
        print(f"  Saved features to {feat_path.name} ({size_mb:.1f} MB)")

    # Cleanup
    del model_lora, model_fresh
    torch.cuda.empty_cache()

    pd.DataFrame(all_results).to_csv(
        Path(CONFIG["output_dir"]) / "per_fold_per_class_results.csv", index=False
    )
    print(f"  Fold {fold_idx+1} total: {timedelta(seconds=int(time.time() - fold_start))}")

print(f"\n{'='*70}\nCV COMPLETE — Total: {timedelta(seconds=int(time.time() - overall_start))}\n{'='*70}")

Loading frozen DINOv2 (reused across folds for baseline)...


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main



FOLD 1/3
  Train (LoRA, eligible): 26480
  Train (full, for probe): 26525
  Val: 13262

  --- Frozen baseline ---
  Baseline time: 0:06:33

  --- DINOv2 + LoRA + linear probe ---
  Reloading fresh DINOv2 for clean LoRA application...


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


    Reload took 36.8s
trainable params: 3,932,160 || all params: 308,300,800 || trainable%: 1.2754
    Fold 1 Ep 1 batch 0/414: loss=2.6321
    Fold 1 Ep 1 batch 100/414: loss=1.1563
    Fold 1 Ep 1 batch 200/414: loss=0.6985
    Fold 1 Ep 1 batch 300/414: loss=0.4044
    Fold 1 Ep 1 batch 400/414: loss=0.4447
    Fold 1 Ep 1: avg loss=0.8537 (9.3 min)
    Fold 1 Ep 2 batch 0/414: loss=0.2964
    Fold 1 Ep 2 batch 100/414: loss=0.4325
    Fold 1 Ep 2 batch 200/414: loss=0.2309
    Fold 1 Ep 2 batch 300/414: loss=0.1537
    Fold 1 Ep 2 batch 400/414: loss=0.1939
    Fold 1 Ep 2: avg loss=0.2392 (9.3 min)
  LoRA time: 0:26:51
  Saved features to features_fold0.npz (286.2 MB)
  Fold 1 total: 0:33:36

FOLD 2/3
  Train (LoRA, eligible): 26479
  Train (full, for probe): 26526
  Val: 13261

  --- Frozen baseline ---
  Baseline time: 0:06:45

  --- DINOv2 + LoRA + linear probe ---
  Reloading fresh DINOv2 for clean LoRA application...


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


    Reload took 45.8s
trainable params: 3,932,160 || all params: 308,300,800 || trainable%: 1.2754
    Fold 2 Ep 1 batch 0/414: loss=2.6231
    Fold 2 Ep 1 batch 100/414: loss=0.9567
    Fold 2 Ep 1 batch 200/414: loss=0.8617
    Fold 2 Ep 1 batch 300/414: loss=0.7377
    Fold 2 Ep 1 batch 400/414: loss=0.7367
    Fold 2 Ep 1: avg loss=0.9654 (9.4 min)
    Fold 2 Ep 2 batch 0/414: loss=0.8034
    Fold 2 Ep 2 batch 100/414: loss=0.6774
    Fold 2 Ep 2 batch 200/414: loss=0.6116
    Fold 2 Ep 2 batch 300/414: loss=0.5432
    Fold 2 Ep 2 batch 400/414: loss=0.3907
    Fold 2 Ep 2: avg loss=0.6025 (9.4 min)
  LoRA time: 0:26:48
  Saved features to features_fold1.npz (286.2 MB)
  Fold 2 total: 0:33:46

FOLD 3/3
  Train (LoRA, eligible): 26479
  Train (full, for probe): 26526
  Val: 13261

  --- Frozen baseline ---
  Baseline time: 0:06:37

  --- DINOv2 + LoRA + linear probe ---
  Reloading fresh DINOv2 for clean LoRA application...


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


    Reload took 55.4s
trainable params: 3,932,160 || all params: 308,300,800 || trainable%: 1.2754
    Fold 3 Ep 1 batch 0/414: loss=2.6596
    Fold 3 Ep 1 batch 100/414: loss=0.8312
    Fold 3 Ep 1 batch 200/414: loss=0.7311
    Fold 3 Ep 1 batch 300/414: loss=0.8608
    Fold 3 Ep 1 batch 400/414: loss=0.3595
    Fold 3 Ep 1: avg loss=0.9078 (9.4 min)
    Fold 3 Ep 2 batch 0/414: loss=0.4013
    Fold 3 Ep 2 batch 100/414: loss=0.2151
    Fold 3 Ep 2 batch 200/414: loss=0.3302
    Fold 3 Ep 2 batch 300/414: loss=0.2828
    Fold 3 Ep 2 batch 400/414: loss=0.2702
    Fold 3 Ep 2: avg loss=0.2721 (9.5 min)
  LoRA time: 0:26:54
  Saved features to features_fold2.npz (286.6 MB)
  Fold 3 total: 0:33:43

CV COMPLETE — Total: 1:41:48


## Aggregate results: per-class mean and std across folds

In [10]:
results_df = pd.DataFrame(all_results)

summary_rows = []
for method in ["frozen_linear_probe", "lora_linear_probe"]:
    for cls_name in classnames:
        class_data = results_df[(results_df["method"] == method) & (results_df["class"] == cls_name)]
        summary_rows.append({
            "method": method, "class": cls_name,
            "support_total": int(class_data["support"].sum()),
            "f1_mean": round(class_data["f1"].mean(), 4),
            "f1_std": round(class_data["f1"].std(ddof=1), 4),
            "precision_mean": round(class_data["precision"].mean(), 4),
            "precision_std": round(class_data["precision"].std(ddof=1), 4),
            "recall_mean": round(class_data["recall"].mean(), 4),
            "recall_std": round(class_data["recall"].std(ddof=1), 4),
            "trained_in_lora": cls_name in eligible_class_set,
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(Path(CONFIG["output_dir"]) / "per_class_summary_mean_std.csv", index=False)
summary_df

,method,class,support_total,f1_mean,f1_std,precision_mean,precision_std,recall_mean,recall_std,trained_in_lora
0,frozen_linear_probe,Araneae,564,0.9929,0.0016,0.9965,0.0061,0.9894,0.0054,True
1,frozen_linear_probe,Blattodea,23,0.9361,0.0625,0.9583,0.0722,0.9167,0.0722,False
2,frozen_linear_probe,Coleoptera,1143,0.8634,0.0170,0.9125,0.0109,0.8198,0.0324,True
3,frozen_linear_probe,Diptera,31913,0.9881,0.0010,0.9827,0.0006,0.9937,0.0016,True
4,frozen_linear_probe,Hemiptera,775,0.8430,0.0070,0.9128,0.0050,0.7832,0.0144,True
5,frozen_linear_probe,Hymenoptera,3326,0.9391,0.0019,0.9463,0.0064,0.9321,0.0051,True
6,frozen_linear_probe,Ixodida,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,False
7,frozen_linear_probe,Lepidoptera,1744,0.9853,0.0030,0.9885,0.0039,0.9823,0.0052,True
8,frozen_linear_probe,Mecoptera,34,0.1545,0.0119,0.6667,0.2887,0.0884,0.0044,False
9,frozen_linear_probe,Neuroptera,115,0.6804,0.0671,0.8013,0.0824,0.5918,0.0607,True


## Headline aggregate metrics

In [11]:
headline_rows = []
for method in ["frozen_linear_probe", "lora_linear_probe"]:
    per_fold_weighted = []
    per_fold_macro_all = []
    per_fold_macro_eligible = []
    per_fold_hemi = []
    per_fold_cole = []

    for fold in range(CONFIG["k_folds"]):
        fold_data = results_df[(results_df["method"] == method) & (results_df["fold"] == fold)]
        per_fold_weighted.append(np.average(fold_data["f1"], weights=fold_data["support"]))
        per_fold_macro_all.append(fold_data["f1"].mean())
        eligible_data = fold_data[fold_data["class"].isin(eligible_class_set)]
        per_fold_macro_eligible.append(eligible_data["f1"].mean())
        per_fold_hemi.append(fold_data[fold_data["class"] == "Hemiptera"]["f1"].iloc[0])
        per_fold_cole.append(fold_data[fold_data["class"] == "Coleoptera"]["f1"].iloc[0])

    headline_rows.append({
        "method": method,
        "weighted_f1_mean": round(np.mean(per_fold_weighted), 4),
        "weighted_f1_std": round(np.std(per_fold_weighted, ddof=1), 4),
        "macro_f1_all_mean": round(np.mean(per_fold_macro_all), 4),
        "macro_f1_all_std": round(np.std(per_fold_macro_all, ddof=1), 4),
        "macro_f1_eligible_mean": round(np.mean(per_fold_macro_eligible), 4),
        "macro_f1_eligible_std": round(np.std(per_fold_macro_eligible, ddof=1), 4),
        "Hemiptera_f1_mean": round(np.mean(per_fold_hemi), 4),
        "Hemiptera_f1_std": round(np.std(per_fold_hemi, ddof=1), 4),
        "Coleoptera_f1_mean": round(np.mean(per_fold_cole), 4),
        "Coleoptera_f1_std": round(np.std(per_fold_cole, ddof=1), 4),
    })

headline_df = pd.DataFrame(headline_rows)
headline_df.to_csv(Path(CONFIG["output_dir"]) / "headline_metrics_mean_std.csv", index=False)
headline_df

,method,weighted_f1_mean,weighted_f1_std,macro_f1_all_mean,macro_f1_all_std,macro_f1_eligible_mean,macro_f1_eligible_std,Hemiptera_f1_mean,Hemiptera_f1_std,Coleoptera_f1_mean,Coleoptera_f1_std
0,frozen_linear_probe,0.9756,0.0011,0.7286,0.0026,0.9011,0.0077,0.8430,0.0070,0.8634,0.0170
1,lora_linear_probe,0.9638,0.0236,0.5953,0.1601,0.7939,0.1344,0.7603,0.1604,0.8286,0.1249


In [12]:
## Per-class delta (LoRA - Frozen)

In [13]:
delta_rows = []
for cls_name in classnames:
    f = summary_df[(summary_df["method"] == "frozen_linear_probe") & (summary_df["class"] == cls_name)]
    l = summary_df[(summary_df["method"] == "lora_linear_probe") & (summary_df["class"] == cls_name)]
    delta_rows.append({
        "class": cls_name,
        "support": int(f["support_total"].iloc[0]),
        "frozen_f1_mean": f["f1_mean"].iloc[0],
        "frozen_f1_std": f["f1_std"].iloc[0],
        "lora_f1_mean": l["f1_mean"].iloc[0],
        "lora_f1_std": l["f1_std"].iloc[0],
        "delta_mean": round(l["f1_mean"].iloc[0] - f["f1_mean"].iloc[0], 4),
        "trained_in_lora": cls_name in eligible_class_set,
    })
delta_df = pd.DataFrame(delta_rows)
delta_df.to_csv(Path(CONFIG["output_dir"]) / "per_class_delta.csv", index=False)
delta_df

,class,support,frozen_f1_mean,frozen_f1_std,lora_f1_mean,lora_f1_std,delta_mean,trained_in_lora
0,Araneae,564,0.9929,0.0016,0.9539,0.0639,-0.0390,True
1,Blattodea,23,0.9361,0.0625,0.5421,0.4715,-0.3940,False
2,Coleoptera,1143,0.8634,0.0170,0.8286,0.1249,-0.0348,True
3,Diptera,31913,0.9881,0.0010,0.9838,0.0097,-0.0043,True
4,Hemiptera,775,0.8430,0.0070,0.7603,0.1604,-0.0827,True
5,Hymenoptera,3326,0.9391,0.0019,0.9042,0.0694,-0.0349,True
6,Ixodida,0,0.0000,0.0000,0.0000,0.0000,0.0000,False
7,Lepidoptera,1744,0.9853,0.0030,0.9760,0.0125,-0.0093,True
8,Mecoptera,34,0.1545,0.0119,0.0476,0.0825,-0.1069,False
9,Neuroptera,115,0.6804,0.0671,0.4932,0.2269,-0.1872,True


## Output files summary

| File | Purpose |
|---|---|
| `per_fold_per_class_results.csv` | Granular per-fold per-class precision/recall/F1 |
| `per_class_summary_mean_std.csv` | Per-class aggregates with mean ± std across folds |
| `per_class_delta.csv` | LoRA vs Frozen per-class deltas |
| `headline_metrics_mean_std.csv` | Top-line aggregates with mean ± std |
| `features_fold{0,1,2}.npz` | Cached features per fold |

Goal: Comparing with `output_cv_lora/` (BioCLIP 2 + LoRA results) to determine whether:
- LoRA adaptation provides similar gains on diagnostic classes regardless of pretraining domain
- BioCLIP 2's biological pretraining advantage is realized through frozen features, through LoRA adaptation, or both
- DINOv2 + LoRA matches, exceeds, or underperforms BioCLIP 2 + LoRA after adaptation


In [14]:
data = np.load(Path(CONFIG["output_dir"]) / "features_fold0.npz")
val_lora = data["val_features_lora"]
val_frozen = data["val_features_frozen"]

print("Frozen features stats:")
print(f"  Mean norm: {np.linalg.norm(val_frozen, axis=1).mean():.4f}")
print(f"  Mean: {val_frozen.mean():.4f}, Std: {val_frozen.std():.4f}")

print("\nLoRA features stats:")
print(f"  Mean norm: {np.linalg.norm(val_lora, axis=1).mean():.4f}")
print(f"  Mean: {val_lora.mean():.4f}, Std: {val_lora.std():.4f}")

Frozen features stats:
  Mean norm: 1.0000
  Mean: -0.0001, Std: 0.0312

LoRA features stats:
  Mean norm: 1.0000
  Mean: -0.0000, Std: 0.0312


In [15]:
data = np.load(Path(CONFIG["output_dir"]) / "features_fold0.npz")
val_lora = data["val_features_lora"]
val_frozen = data["val_features_frozen"]
val_labels = data["val_labels"]

# Cosine similarity per-example: frozen vs LoRA
from sklearn.metrics.pairwise import cosine_similarity as cos_sim_pairwise

# Per-example: how similar is frozen vs LoRA feature for the SAME image?
per_example_sim = np.array([
    np.dot(val_frozen[i], val_lora[i]) 
    for i in range(len(val_frozen))
])
print(f"Per-example cosine similarity (frozen vs LoRA, same image):")
print(f"  Mean: {per_example_sim.mean():.4f}")
print(f"  Std: {per_example_sim.std():.4f}")
print(f"  Min: {per_example_sim.min():.4f}")
print(f"  Max: {per_example_sim.max():.4f}")

# Class separability
from sklearn.metrics import silhouette_score
sample_idx = np.random.choice(len(val_labels), 3000, replace=False)
sil_frozen = silhouette_score(val_frozen[sample_idx], val_labels[sample_idx], metric='cosine')
sil_lora = silhouette_score(val_lora[sample_idx], val_labels[sample_idx], metric='cosine')
print(f"\nSilhouette score (higher = better class separation):")
print(f"  Frozen: {sil_frozen:.4f}")
print(f"  LoRA:   {sil_lora:.4f}")

Per-example cosine similarity (frozen vs LoRA, same image):
  Mean: 0.1241
  Std: 0.0486
  Min: -0.0207
  Max: 0.3599

Silhouette score (higher = better class separation):
  Frozen: 0.1199
  LoRA:   0.7982
